# Metadata Configuration

This notebook creates and populates the metadata control tables that drive the entire pipeline. These tables define source connections, table configurations, transformation rules, and data quality expectations — all stored as data, not hardcoded in pipeline code.

In [0]:
# ============================================================================
# Create all 5 metadata control tables (no DEFAULT clauses for Delta compat)
# ============================================================================
CATALOG = "metadata_framework"
SCHEMA = f"{CATALOG}.bronze"

# 1. source_config — defines external data sources
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.source_config (
  source_id          STRING   NOT NULL COMMENT 'Unique source system identifier',
  source_name        STRING   NOT NULL COMMENT 'Human-readable source name',
  source_type        STRING   NOT NULL COMMENT 'csv, json, parquet, delta, jdbc, api',
  source_path        STRING   NOT NULL COMMENT 'Fully-qualified table name or file path',
  connection_details STRING   COMMENT 'JSON string with connection parameters',
  is_active          BOOLEAN  NOT NULL COMMENT 'Whether this source is active',
  created_at         TIMESTAMP NOT NULL COMMENT 'Creation timestamp',
  updated_at         TIMESTAMP NOT NULL COMMENT 'Last update timestamp'
) COMMENT 'Defines external data sources for the pipeline.'
""")

# 2. table_config — defines tables to ingest and their target locations
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.table_config (
  table_id           STRING   NOT NULL COMMENT 'Unique table configuration identifier',
  source_id          STRING   NOT NULL COMMENT 'FK to source_config',
  source_table_name  STRING   NOT NULL COMMENT 'Source table or file name',
  target_schema      STRING   NOT NULL COMMENT 'Target schema: bronze, silver, gold',
  target_table_name  STRING   NOT NULL COMMENT 'Target table name',
  load_type          STRING   NOT NULL COMMENT 'full_load, incremental, cdc',
  incremental_column STRING   COMMENT 'Column for incremental loads',
  primary_keys       STRING   COMMENT 'Comma-separated PK columns for merge',
  partition_columns STRING   COMMENT 'Comma-separated partition columns',
  is_active          BOOLEAN  NOT NULL COMMENT 'Whether this config is active',
  created_at         TIMESTAMP NOT NULL COMMENT 'Creation timestamp',
  updated_at         TIMESTAMP NOT NULL COMMENT 'Last update timestamp'
) COMMENT 'Defines table-level ingestion configurations.'
""")

# 3. transformation_rules — column-level transformations (bronze → silver)
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.transformation_rules (
  rule_id            STRING   NOT NULL COMMENT 'Unique transformation rule identifier',
  table_id           STRING   NOT NULL COMMENT 'FK to table_config',
  source_column      STRING   NOT NULL COMMENT 'Source column from bronze',
  target_column      STRING   NOT NULL COMMENT 'Target column in silver',
  transformation_sql STRING   NOT NULL COMMENT 'SQL expression to apply',
  data_type          STRING   NOT NULL COMMENT 'STRING, INT, DOUBLE, DATE, TIMESTAMP, BOOLEAN',
  is_nullable        BOOLEAN  NOT NULL COMMENT 'Whether target allows nulls',
  execution_order    INT      NOT NULL COMMENT 'Order rules are applied',
  is_active          BOOLEAN  NOT NULL COMMENT 'Whether rule is active',
  created_at         TIMESTAMP NOT NULL COMMENT 'Creation timestamp',
  updated_at         TIMESTAMP NOT NULL COMMENT 'Last update timestamp'
) COMMENT 'Defines column-level transformation rules.'
""")

# 4. data_quality_rules — DQ checks per table/column
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.data_quality_rules (
  dq_rule_id         STRING   NOT NULL COMMENT 'Unique DQ rule identifier',
  table_id           STRING   NOT NULL COMMENT 'FK to table_config',
  rule_name          STRING   NOT NULL COMMENT 'Human-readable quality check name',
  rule_type          STRING   NOT NULL COMMENT 'not_null, unique, range, regex, custom_sql',
  column_name        STRING   COMMENT 'Column to check (NULL for table-level)',
  rule_expression    STRING   NOT NULL COMMENT 'SQL expression defining the rule',
  severity           STRING   NOT NULL COMMENT 'error, warn, info',
  is_active          BOOLEAN  NOT NULL COMMENT 'Whether rule is active',
  created_at         TIMESTAMP NOT NULL COMMENT 'Creation timestamp',
  updated_at         TIMESTAMP NOT NULL COMMENT 'Last update timestamp'
) COMMENT 'Defines data quality validation rules.'
""")

# 5. pipeline_run_log — tracks each pipeline execution run
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SCHEMA}.pipeline_run_log (
  run_id           STRING    NOT NULL COMMENT 'Unique run identifier (UUID)',
  table_id         STRING    COMMENT 'FK to table_config (NULL for full pipeline runs)',
  run_status       STRING    NOT NULL COMMENT 'started, running, success, failed, skipped',
  run_layer        STRING    NOT NULL COMMENT 'bronze, silver, gold',
  records_read     BIGINT    COMMENT 'Records read from source',
  records_written  BIGINT    COMMENT 'Records written to target',
  records_rejected BIGINT    COMMENT 'Records rejected by DQ checks',
  error_message    STRING    COMMENT 'Error message if failed',
  started_at       TIMESTAMP NOT NULL COMMENT 'Run start time',
  completed_at     TIMESTAMP  COMMENT 'Run completion time'
) COMMENT 'Tracks pipeline execution history.'
""")

print("All 5 metadata tables created (or already exist).")

In [0]:
# ============================================================================
# Seed source_config & table_config for all 4 source tables
# Uses Python DataFrames + temp views + MERGE for idempotent re-runs
# ============================================================================
from datetime import datetime
_now = datetime.now()

# --- source_config ---
_sc = [
    ("src_customers", "Customer Source (Delta)", "delta", "metadata_framework.source.customers", True,  _now, _now),
    ("src_orders",    "Order Source (Delta)",    "delta", "metadata_framework.source.orders",    True,  _now, _now),
    ("src_products",  "Product Source (Delta)",  "delta", "metadata_framework.source.products",  True,  _now, _now),
    ("src_payments",  "Payment Source (Delta)",  "delta", "metadata_framework.source.payments",  True,  _now, _now),
]
spark.createDataFrame(_sc, schema="source_id STRING, source_name STRING, source_type STRING, source_path STRING, is_active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP").createOrReplaceTempView("_sc_stg")

spark.sql(f"""
MERGE INTO {SCHEMA}.source_config AS t
USING _sc_stg AS s
ON t.source_id = s.source_id
WHEN MATCHED THEN UPDATE SET source_name=s.source_name, source_type=s.source_type, source_path=s.source_path, is_active=s.is_active, updated_at=current_timestamp()
WHEN NOT MATCHED THEN INSERT (source_id, source_name, source_type, source_path, is_active, created_at, updated_at)
VALUES (s.source_id, s.source_name, s.source_type, s.source_path, s.is_active, s.created_at, s.updated_at)
""")

# --- table_config ---
_tc = [
    ("tbl_customers", "src_customers", "customers", "bronze", "raw_customers", "full_load",   None,           "customer_id", None,          True, _now, _now),
    ("tbl_orders",    "src_orders",    "orders",    "bronze", "raw_orders",    "incremental", "order_date",   "order_id",    "order_date",  True, _now, _now),
    ("tbl_products",  "src_products", "products",  "bronze", "raw_products",  "full_load",   None,           "product_id",  None,          True, _now, _now),
    ("tbl_payments",  "src_payments", "payments",  "bronze", "raw_payments",  "incremental", "payment_date", "payment_id",  None,          True, _now, _now),
]
spark.createDataFrame(_tc, schema="table_id STRING, source_id STRING, source_table_name STRING, target_schema STRING, target_table_name STRING, load_type STRING, incremental_column STRING, primary_keys STRING, partition_columns STRING, is_active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP").createOrReplaceTempView("_tc_stg")

spark.sql(f"""
MERGE INTO {SCHEMA}.table_config AS t
USING _tc_stg AS s
ON t.table_id = s.table_id
WHEN MATCHED THEN UPDATE SET source_id=s.source_id, source_table_name=s.source_table_name, target_schema=s.target_schema, target_table_name=s.target_table_name, load_type=s.load_type, incremental_column=s.incremental_column, primary_keys=s.primary_keys, partition_columns=s.partition_columns, is_active=s.is_active, updated_at=current_timestamp()
WHEN NOT MATCHED THEN INSERT (table_id, source_id, source_table_name, target_schema, target_table_name, load_type, incremental_column, primary_keys, partition_columns, is_active, created_at, updated_at)
VALUES (s.table_id, s.source_id, s.source_table_name, s.target_schema, s.target_table_name, s.load_type, s.incremental_column, s.primary_keys, s.partition_columns, s.is_active, s.created_at, s.updated_at)
""")

print("source_config & table_config seeded.")

In [0]:
# ============================================================================
# Seed transformation_rules for all 4 tables (32 column rules)
# ============================================================================
_tr = [
    # Customers (8)
    ("rule_cust_1", "tbl_customers", "customer_id",       "customer_id",       "customer_id",                   "STRING", False, 1, True, _now, _now),
    ("rule_cust_2", "tbl_customers", "customer_name",     "customer_name",     "TRIM(customer_name)",           "STRING", True,  2, True, _now, _now),
    ("rule_cust_3", "tbl_customers", "email",             "email",             "LOWER(TRIM(email))",            "STRING", True,  3, True, _now, _now),
    ("rule_cust_4", "tbl_customers", "phone",             "phone",             "phone",                        "STRING", True,  4, True, _now, _now),
    ("rule_cust_5", "tbl_customers", "address",           "address",           "address",                      "STRING", True,  5, True, _now, _now),
    ("rule_cust_6", "tbl_customers", "city",              "city",              "city",                         "STRING", True,  6, True, _now, _now),
    ("rule_cust_7", "tbl_customers", "country",           "country",           "country",                      "STRING", True,  7, True, _now, _now),
    ("rule_cust_8", "tbl_customers", "registration_date", "registration_date", "CAST(registration_date AS DATE)", "DATE", True, 8, True, _now, _now),
    # Orders (8)
    ("rule_ord_1", "tbl_orders", "order_id",      "order_id",      "order_id",                     "STRING", False, 1, True, _now, _now),
    ("rule_ord_2", "tbl_orders", "customer_id",   "customer_id",   "customer_id",                  "STRING", False, 2, True, _now, _now),
    ("rule_ord_3", "tbl_orders", "product_id",    "product_id",    "product_id",                   "STRING", True,  3, True, _now, _now),
    ("rule_ord_4", "tbl_orders", "order_date",    "order_date",    "CAST(order_date AS DATE)",      "DATE",   False, 4, True, _now, _now),
    ("rule_ord_5", "tbl_orders", "quantity",      "quantity",      "CAST(quantity AS INT)",        "INT",    True,  5, True, _now, _now),
    ("rule_ord_6", "tbl_orders", "unit_price",    "unit_price",    "CAST(unit_price AS DOUBLE)",   "DOUBLE", True,  6, True, _now, _now),
    ("rule_ord_7", "tbl_orders", "total_amount",  "total_amount",  "CAST(total_amount AS DOUBLE)", "DOUBLE", True,  7, True, _now, _now),
    ("rule_ord_8", "tbl_orders", "order_status",  "order_status",  "order_status",                 "STRING", True,  8, True, _now, _now),
    # Products (8)
    ("rule_prod_1", "tbl_products", "product_id",     "product_id",     "product_id",                    "STRING", False, 1, True, _now, _now),
    ("rule_prod_2", "tbl_products", "product_name",   "product_name",   "TRIM(product_name)",           "STRING", True,  2, True, _now, _now),
    ("rule_prod_3", "tbl_products", "category",       "category",       "category",                      "STRING", True,  3, True, _now, _now),
    ("rule_prod_4", "tbl_products", "unit_price",    "unit_price",    "CAST(unit_price AS DOUBLE)",   "DOUBLE", True,  4, True, _now, _now),
    ("rule_prod_5", "tbl_products", "cost_price",    "cost_price",    "CAST(cost_price AS DOUBLE)",   "DOUBLE", True,  5, True, _now, _now),
    ("rule_prod_6", "tbl_products", "stock_quantity", "stock_quantity", "CAST(stock_quantity AS INT)", "INT",    True,  6, True, _now, _now),
    ("rule_prod_7", "tbl_products", "supplier",      "supplier",      "supplier",                      "STRING", True,  7, True, _now, _now),
    ("rule_prod_8", "tbl_products", "launch_date",   "launch_date",   "CAST(launch_date AS DATE)",     "DATE",   True,  8, True, _now, _now),
    # Payments (8)
    ("rule_pay_1", "tbl_payments", "payment_id",      "payment_id",      "payment_id",                     "STRING", False, 1, True, _now, _now),
    ("rule_pay_2", "tbl_payments", "order_id",        "order_id",        "order_id",                       "STRING", False, 2, True, _now, _now),
    ("rule_pay_3", "tbl_payments", "customer_id",     "customer_id",     "customer_id",                    "STRING", False, 3, True, _now, _now),
    ("rule_pay_4", "tbl_payments", "payment_method",  "payment_method",  "payment_method",                 "STRING", True,  4, True, _now, _now),
    ("rule_pay_5", "tbl_payments", "payment_amount",  "payment_amount",  "CAST(payment_amount AS DOUBLE)", "DOUBLE", True,  5, True, _now, _now),
    ("rule_pay_6", "tbl_payments", "payment_date",    "payment_date",    "CAST(payment_date AS DATE)",     "DATE",   True,  6, True, _now, _now),
    ("rule_pay_7", "tbl_payments", "payment_status",  "payment_status",  "payment_status",                 "STRING", True,  7, True, _now, _now),
    ("rule_pay_8", "tbl_payments", "transaction_ref", "transaction_ref", "transaction_ref",               "STRING", True,  8, True, _now, _now),
]
_tr_schema = "rule_id STRING, table_id STRING, source_column STRING, target_column STRING, transformation_sql STRING, data_type STRING, is_nullable BOOLEAN, execution_order INT, is_active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
spark.createDataFrame(_tr, schema=_tr_schema).createOrReplaceTempView("_tr_stg")

spark.sql(f"""
MERGE INTO {SCHEMA}.transformation_rules AS t
USING _tr_stg AS s
ON t.rule_id = s.rule_id
WHEN MATCHED THEN UPDATE SET * EXCEPT (created_at)
WHEN NOT MATCHED THEN INSERT *
""")

print(f"transformation_rules seeded: {len(_tr)} rules across 4 tables.")

In [0]:
# ============================================================================
# Seed data_quality_rules for all 4 tables (12 DQ checks)
# ============================================================================
_dq = [
    # Customers
    ("dq_cust_1", "tbl_customers", "Customer ID not null", "not_null", "customer_id", "IS NOT NULL",                         "error", True, _now, _now),
    ("dq_cust_2", "tbl_customers", "Customer ID unique",   "unique",   "customer_id", "COUNT(*) = COUNT(DISTINCT customer_id)", "error", True, _now, _now),
    ("dq_cust_3", "tbl_customers", "Email not null",      "not_null", "email",       "IS NOT NULL",                         "warn",  True, _now, _now),
    # Orders
    ("dq_ord_1",  "tbl_orders", "Order ID not null",  "not_null", "order_id",      "IS NOT NULL",                      "error", True, _now, _now),
    ("dq_ord_2",  "tbl_orders", "Order ID unique",    "unique",   "order_id",      "COUNT(*) = COUNT(DISTINCT order_id)", "error", True, _now, _now),
    ("dq_ord_3",  "tbl_orders", "Total amount > 0",   "range",    "total_amount",  "> 0",                              "warn",  True, _now, _now),
    # Products
    ("dq_prod_1", "tbl_products", "Product ID not null", "not_null", "product_id", "IS NOT NULL",                        "error", True, _now, _now),
    ("dq_prod_2", "tbl_products", "Product ID unique",   "unique",   "product_id", "COUNT(*) = COUNT(DISTINCT product_id)", "error", True, _now, _now),
    ("dq_prod_3", "tbl_products", "Unit price > 0",     "range",    "unit_price",  "> 0",                                "error", True, _now, _now),
    # Payments
    ("dq_pay_1",  "tbl_payments", "Payment ID not null", "not_null", "payment_id",     "IS NOT NULL",                           "error", True, _now, _now),
    ("dq_pay_2",  "tbl_payments", "Payment ID unique",   "unique",   "payment_id",     "COUNT(*) = COUNT(DISTINCT payment_id)", "error", True, _now, _now),
    ("dq_pay_3",  "tbl_payments", "Payment amount > 0", "range",    "payment_amount", "> 0",                                   "error", True, _now, _now),
]
_dq_schema = "dq_rule_id STRING, table_id STRING, rule_name STRING, rule_type STRING, column_name STRING, rule_expression STRING, severity STRING, is_active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
spark.createDataFrame(_dq, schema=_dq_schema).createOrReplaceTempView("_dq_stg")

spark.sql(f"""
MERGE INTO {SCHEMA}.data_quality_rules AS t
USING _dq_stg AS s
ON t.dq_rule_id = s.dq_rule_id
WHEN MATCHED THEN UPDATE SET * EXCEPT (created_at)
WHEN NOT MATCHED THEN INSERT *
""")

print(f"data_quality_rules seeded: {len(_dq)} rules across 4 tables.")
print("\nMetadata configuration complete!")
print(f"  source_config:        {spark.read.table(f'{SCHEMA}.source_config').count()} rows")
print(f"  table_config:         {spark.read.table(f'{SCHEMA}.table_config').count()} rows")
print(f"  transformation_rules: {spark.read.table(f'{SCHEMA}.transformation_rules').count()} rows")
print(f"  data_quality_rules:   {spark.read.table(f'{SCHEMA}.data_quality_rules').count()} rows")